# index-by-tensor — ex2: advanced indexing with TWO index tensors (rows + cols) for paired gather

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `index-by-tensor`. Running the final beacon cell reports progress against the `PyTorch: index by tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: index by tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`index-by-tensor`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "index-by-tensor"
DD_SUBTOPIC = "PyTorch: index by tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Advanced indexing with two index tensors (rows + cols)

Ex1 used ONE index tensor for the leading axis (`embed[idx]`). The deepening move is two index tensors broadcasting against each other for explicit gather:

```python
x = t.arange(20).reshape(4, 5)              # (4, 5)
rows = t.tensor([0, 2, 3])                  # (3,)
cols = t.tensor([1, 4, 0])                  # (3,)
x[rows, cols]                                # (3,) — diagonal pick
# → tensor([x[0,1], x[2,4], x[3,0]])
```

**Both index tensors broadcast together to form the OUTPUT shape.** When `rows` and `cols` are both shape `(K,)`, the result is shape `(K,)` and you get K paired picks. To produce a `(R, C)` grid instead, reshape: `x[rows[:, None], cols[None, :]]`.

**Why advanced indexing over `gather`.** `gather(dim, idx)` requires the index tensor to broadcast against the input on ALL axes except `dim`. Multi-axis advanced indexing has cleaner semantics when you have one index per output element — it's the 'pick K specific cells' operation in shorthand.

**Shape rule recap.** Multiple index tensors → output shape is the broadcast of those index tensors' shapes. The original axes being indexed COLLAPSE. So `(4, 5)` indexed by two `(K,)` tensors gives a `(K,)` output, not `(K, 5)`.

### Exercise 2 — advanced indexing with TWO index tensors (rows + cols) for paired gather

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply two-index-tensor advanced indexing both in PAIRED form `x[rows, cols] → (K,)` and in GRID form `x[rows[:, None], cols[None, :]] → (R, C)`.
> Keywords: advanced-indexing, gather, two-index-tensors, broadcasting
> ```

**KCs targeted:** `paired-index-collapse-axes`, `broadcasted-grid-via-newaxis`

Implement `ex2_gather_with_two_indices(x, rows, cols)`.

Inputs:
- `x`: `(H, W)` 2-D tensor.
- `rows`: `(K,)` LongTensor with values in `[0, H)`.
- `cols`: `(K,)` LongTensor with values in `[0, W)`.

Return a dict with TWO outputs of the same data, in two different output shapes:
```
{
  'paired': x[rows, cols],                       # (K,)
  'grid':   x[rows[:, None], cols[None, :]],     # (K, K)
}
```

- `paired[i] == x[rows[i], cols[i]]` — paired (zip-style) indexing.
- `grid[i, j] == x[rows[i], cols[j]]` — Cartesian-product indexing.

Constraints:
- DO NOT use a Python for-loop.
- DO NOT use `torch.gather` — exercise the advanced-indexing syntax directly.
- Both outputs must preserve `x.dtype`.

In [ ]:
def ex2_gather_with_two_indices(x, rows, cols):
    return {
        'paired': x[rows, cols],
        'grid': x[rows[:, None], cols[None, :]],
    }


<details><summary>Solution</summary>

```python
def ex2_gather_with_two_indices(x, rows, cols):
    return {
        'paired': x[rows, cols],
        'grid': x[rows[:, None], cols[None, :]],
    }
```

**Paired vs grid is a SHAPE decision, not a values decision.** Same input tensors. Different bracket syntax. The first broadcasts `rows` and `cols` together (both `(K,)` → output `(K,)`). The second reshapes them into `(K, 1)` and `(1, K)` so they broadcast to `(K, K)` — and the output picks up that shape.

**`rows[:, None]` is `rows.unsqueeze(1)`.** Same operation, shorter notation. Use whichever your team's style guide prefers. The `None` form reads more like NumPy.

**Why this is the canonical bilinear-interp building block.** Image warping (grid_sample, STN, RoI pooling) all reduce to 'gather these four corner pixels per output pixel'. The `x[rows[:, None], cols[None, :]]` pattern is the rank-2 version; rank-4 extensions use the same idea with more broadcasting axes.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()